In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict,Annotated
import os
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import requests
import random
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

ImportError: cannot import name 'tools' from 'langchain_core.tools' (c:\Users\Kashish\Downloads\UPCON26_PHP (5)\Agentic-AI\env\Lib\site-packages\langchain_core\tools\__init__.py)

In [ ]:
endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )
model=ChatHuggingFace(llm=endpoint)

In [ ]:
#tools

search_tool=DuckDuckGoSearchRun(region="us-en")

@tool
def calculator(first_number:int,second_number:int,operation:str)->dict:
    """Use this to calculate the sum of two numbers"""
    
    
    try:
        if operation=="add":
            result=first_number+second_number
        elif operation=="subtract":
            result=first_number-second_number
        elif operation=="multiply":
            result=first_number*second_number
        elif operation=="divide":
            if second_number==0:
                return {"error":"Cannot divide by zero"}
            result=first_number/second_number
        else:
            return {"error":"Invalid operation"}
        return {"first_number":first_number,"second_number":second_number,"operation":operation,"result":result}
    except Exception as e:
        return {"error":str(e)}
    

@tool
def get_stock_price(ticker:str)->dict:
    url:f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={ticker}&apikey={os.getenv('ALPHA_VANTAGE_API_KEY')}"
    r=requests.get(url)
    return r.json()


ImportError: Could not import ddgs python package. Please install it with `pip install -U ddgs`.

In [ ]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]


def chat_node(state:ChatState):
    messages=state["messages"]
    response=model.invoke(messages)
    return {"messages":[response]}

tool_node=ToolNode(tools)

NameError: name 'tools' is not defined

In [ ]:
graph=StateGraph(ChatState)
graph.add_node("chat",chat_node)
graph.add_node("tool",tool_node)

graph.add_edge(START,chat_node)
graph.add_conditional_edges(
    "chat_node",
    tools_condition
)

graph=graph.compile()

NameError: name 'ChatState' is not defined